# temperature — 같은 질문, 매번 같은 답일까?

`1_1_openai` 에서 본 호출은 `model` 과 `messages` 만 넘겼습니다.
여기에 **딱 하나의 숫자** `temperature` 를 더하면 모델의 성격이 바뀝니다.

## temperature 한 줄 요약

모델이 다음 단어를 고를 때 **얼마나 모험을 하게 할지** 정하는 손잡이입니다.

| temperature | 모델의 태도 | 결과 |
|---|---|---|
| **0** | 가장 확률 높은 단어만 고른다 | 거의 **항상 같은 답** (정답·요약에 유리) |
| **높음(0.8~1.5)** | 낮은 확률 단어도 집어 든다 | **매번 다른 답** (창작·브레인스토밍에 유리) |

## 이 노트북의 흐름

| 단계 | 내용 |
|---|---|
| **Step 1~2** | 준비 — `1_1` 과 동일 |
| **Step 3~4** | 실험용 질문과 호출 함수 만들기 |
| **Step 5** | `temperature=0` 으로 3번 — 답이 같은지 |
| **Step 6** | `temperature=1.3` 으로 3번 — 답이 달라지는지 |
| **Step 7** | 나란히 놓고 비교 |

> **이 노트북에서 기억할 한 가지**
> temperature 는 "창의성"이 아니라 **답의 흔들림**을 정하는 값입니다.
> 정답이 하나인 일은 낮게, 여러 답이 좋은 일은 높게 씁니다.

## Step 1. 준비물 — `1_1` 과 똑같습니다

라이브러리와 키를 읽어 오는 부분은 `1_1_openai` 와 완전히 같습니다.
자세한 설명은 그 노트북을 참고하세요. 여기서는 빠르게 지나갑니다.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

## Step 2. 키 확인하고 클라이언트 만들기

`.env` 를 올리고, 키가 들어왔는지 앞 8자만 확인한 뒤 클라이언트를 만듭니다.

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

openai = OpenAI()

python-dotenv could not parse statement starting at line 2
python-dotenv could not parse statement starting at line 3


OpenAI API Key exists and begins sk-proj-


## Step 3. 실험용 질문 — 일부러 '답이 여러 개'인 것

temperature 의 차이를 눈으로 보려면 **정답이 하나가 아닌 질문**이 좋습니다.

"3 더하기 5는?" 같은 질문은 temperature 를 올려도 답이 `8` 하나뿐이라 차이가 안 보입니다.
반대로 **표어·이름·문구 짓기**처럼 좋은 답이 수십 개인 일은 흔들림이 선명하게 드러납니다.

```python
user_prompt = "새로 문을 여는 동네 카페의 홍보 문구를 한 줄로 지어 주세요."
```

| 질문 종류 | temperature 효과 |
|---|---|
| 계산·사실 (답 1개) | 거의 안 보인다 |
| 창작·아이디어 (답 여러 개) | **또렷하게 보인다** |

In [3]:
system_prompt = "당신은 감각 있는 카피라이터입니다. 짧고 임팩트 있는 한 줄 문구를 만듭니다."
user_prompt = "새로 문을 여는 동네 카페의 홍보 문구를 한 줄로 지어 주세요."

messages = [{"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}]
messages

[{'role': 'system', 'content': '당신은 감각 있는 카피라이터입니다. 짧고 임팩트 있는 한 줄 문구를 만듭니다.'},
 {'role': 'user', 'content': '새로 문을 여는 동네 카페의 홍보 문구를 한 줄로 지어 주세요.'}]

## Step 4. 호출을 함수로 묶기

같은 호출을 **여러 번, temperature 만 바꿔** 할 것이므로 함수로 만들어 둡니다.

```python
response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    temperature=temperature,   # <- 이번에 새로 추가된 인자
)
```

`1_1` 의 호출과 **딱 한 줄(`temperature=`)만 다릅니다.**

> **모델 선택 주의**: `temperature` 는 `gpt-4.1-mini` 같은 모델에서 동작합니다.
> `gpt-5-nano` 등 일부 최신 모델은 temperature 를 받지 않으니 이 실험에는 `gpt-4.1-mini` 를 씁니다.

In [4]:
def ask(messages, temperature):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

## Step 5. `temperature=0` 으로 3번 물어보기

같은 질문을 **똑같이 세 번** 던집니다. temperature 가 0이면 모델은 매번
가장 확률 높은 단어만 고르므로 **세 답이 거의 똑같이** 나옵니다.

> 완전히 글자 하나까지 같지 않을 수도 있습니다. 0이라도 내부적으로 100% 고정은 아니지만,
> **눈에 띄게 비슷하다**는 것이 핵심입니다.

요약·분류·데이터 추출처럼 **매번 같은 결과가 중요한 일**에 이 설정을 씁니다.

In [5]:
print("=== temperature = 0 (답이 고정되는 쪽) ===\n")
for i in range(3):
    print(f"[{i+1}번째] {ask(messages, temperature=0)}")

=== temperature = 0 (답이 고정되는 쪽) ===

[1번째] “동네 한 모퉁이, 당신의 새로운 커피 쉼터!”
[2번째] “당신의 일상에 향기를 더하는, 우리 동네 작은 카페가 문을 열었어요.”
[3번째] “동네의 작은 쉼표, 당신의 하루를 채우다.”


## Step 6. `temperature=1.3` 으로 3번 물어보기

이번엔 같은 질문에 temperature 만 **높게** 줍니다.
모델이 낮은 확률의 단어도 집어 들기 시작하므로 **세 답이 서로 달라집니다.**

| temperature | 보통 쓰는 곳 |
|---|---|
| 0 ~ 0.3 | 요약, 번역, 분류, 코드 |
| 0.7 ~ 1.0 | 일반 대화, 글쓰기 |
| 1.2 ~ 1.7 | 브레인스토밍, 카피, 아이디어 뽑기 |

> 너무 높이면(예: 1.8+) 문장이 **어색하거나 엉뚱**해질 수 있습니다.
> 창의성과 멀쩡함 사이의 균형점을 찾는 것이 요령입니다.

In [6]:
print("=== temperature = 1.3 (답이 흔들리는 쪽) ===\n")
for i in range(3):
    print(f"[{i+1}번째] {ask(messages, temperature=1.3)}")

=== temperature = 1.3 (답이 흔들리는 쪽) ===

[1번째] “동네 숨켠에서 만나는 당신만의 커피 타임.”
[2번째] 동네의 쉼표, 새로 열린 당신만의 커피 공간!
[3번째] 당신 동네, 행복을 담는 첫 모금 카페!


## Step 7. 나란히 놓고 비교

두 설정의 답을 한 번에 찍어 **차이를 직접 눈으로** 확인합니다.

- 위쪽(0) 세 줄은 서로 **비슷**할 것입니다.
- 아래쪽(1.3) 세 줄은 서로 **제각각**일 것입니다.

같은 코드, 같은 질문, **바뀐 건 숫자 하나뿐**이라는 점이 중요합니다.

In [8]:
print("temperature = 0  (고정)")
for i in range(3):
    print(f"  {i+1}. {ask(messages, temperature=0)}")

print("\ntemperature = 1.3  (흔들림)")
for i in range(3):
    print(f"  {i+1}. {ask(messages, temperature=1.3)}")

temperature = 0  (고정)
  1. “당신의 일상에 향기를 더하는, 우리 동네 작은 카페가 문을 열었어요.”
  2. “당신의 일상에 향기를 더하는, 우리 동네 작은 카페가 문을 열었어요.”
  3. “당신의 일상에 향기를 더하는, 우리 동네 작은 카페가 문을 열었어요.”

temperature = 1.3  (흔들림)
  1. “우리 동네, 당신만의 작은 행복 카페가 열렸어요.”
  2. 동네 담벼락 속, 커피 향 가득한 쉼터가 열렸습니다!
  3. 오늘의 커피, 오늘만의 이야기로 시작하세요.


---

## 정리

`messages` 를 보내는 코드는 그대로 두고 **`temperature` 숫자 하나**만 바꿨습니다.

```
create(model, messages, temperature=0)    ->  매번 비슷한 답
create(model, messages, temperature=1.3)  ->  매번 다른 답
```

### 꼭 기억할 것

| | 내용 |
|---|---|
| **낮으면 고정** | 요약·분류·추출처럼 같은 결과가 중요한 일에 |
| **높으면 흔들림** | 창작·아이디어처럼 다양성이 필요한 일에 |
| **차이는 질문에 달렸다** | 답이 하나뿐인 질문은 temperature 를 올려도 티가 안 난다 |
| **바뀐 건 숫자 하나** | 호출 코드는 `1_1` 과 거의 같다 |

### 직접 해 보기

1. `user_prompt` 를 "대한민국의 수도는?" 같은 **정답이 하나인 질문**으로 바꿔 보세요. temperature 를 올려도 답이 변하나요?
2. `temperature=1.3` 을 `2.0` 으로 올려 보세요. 문장이 어떻게 무너지는지 보세요.
3. `temperature` 대신 `top_p` 라는 인자도 있습니다. 찾아보고 어떤 손잡이인지 비교해 보세요.

### 다음 노트북

| 노트북 | 내용 |
|---|---|
| `1_5_tokenizer` | 모델이 단어를 고른다는 것 — 토큰이 무엇인가 |
| `1_6_stateless` | 모델이 지난 대화를 기억하지 못한다는 것 |